## SRP043546

**paper:** [PMID: 25197823](https://pmc.ncbi.nlm.nih.gov/articles/PMC4157780/) - Comparative Transcriptomic Characterization of the Early Development in Pacific White Shrimp Litopenaeus vannamei, 2014

**date, curator:** 2026-07-29, Sara Carsanaro

**notes**
* some debate over if postlarva is part of larva stage or juvenile stage, current dev stage ontology has it as a part of larva stage so i am annotating it that way for now but made a note to review in the future
* new term request made for mysis stage, see [issue 125](https://github.com/obophenotype/developmental-stage-ontologies/issues/125)

### annotation summary

In [ ]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

In [ ]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

### set variables, import packages, define functions

In [1]:
experiment_id = "SRP043546"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
100%|█████████████████████████████████████████████| 5/5 [00:04<00:00,  1.02it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,,,,6689,,,,,,p_stage,SAMN02870963,,,,,,,"not clear if post larval should be considered final larval stage or first juvenile/sexually immature stage, dev stage states final larval stage. anat annotation should either be sexually immature organism or larva, I will annotate as larva for now however i am making a note in the lineage dev stages to review this, PMID: 25197823",,,,29/07/2026,,,p_stage,,,,,,TRANSCRIPTOMIC,PolyA
1,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,,,,6689,,,,,,m_stage,SAMN02870962,,,,,,,"submitted new term request for mysis stage https://github.com/obophenotype/developmental-stage-ontologies/issues/124, PMID: 25197823",,,,29/07/2026,,,m_stage,,,,,,TRANSCRIPTOMIC,PolyA
2,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,,,,6689,,,,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,,29/07/2026,,,z_stage,,,,,,TRANSCRIPTOMIC,PolyA
3,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,,,,6689,,,,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,,29/07/2026,,,n_stage,,,,,,TRANSCRIPTOMIC,PolyA
4,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,,,,6689,,,,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,,29/07/2026,,,e_stage,,,,,,TRANSCRIPTOMIC,PolyA


#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['whole body']


#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [6]:
unique_sorted(library, "infoStage")

['embryo stage' 'mysis stage' 'nauplius stage' 'postlarvae stage'
 'zoea stage']


#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [7]:
library.loc[:,'sex'] = 'NA'
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,NA,,,6689,,,,,,p_stage,SAMN02870963,,,,,,,"not clear if post larval should be considered final larval stage or first juvenile/sexually immature stage, dev stage states final larval stage. anat annotation should either be sexually immature organism or larva, I will annotate as larva for now however i am making a note in the lineage dev stages to review this, PMID: 25197823",,,,29/07/2026,,,p_stage,,,,,,TRANSCRIPTOMIC,PolyA
1,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,NA,,,6689,,,,,,m_stage,SAMN02870962,,,,,,,"submitted new term request for mysis stage https://github.com/obophenotype/developmental-stage-ontologies/issues/124, PMID: 25197823",,,,29/07/2026,,,m_stage,,,,,,TRANSCRIPTOMIC,PolyA
2,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,NA,,,6689,,,,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,,29/07/2026,,,z_stage,,,,,,TRANSCRIPTOMIC,PolyA
3,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,NA,,,6689,,,,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,,29/07/2026,,,n_stage,,,,,,TRANSCRIPTOMIC,PolyA
4,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,NA,,,6689,,,,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,,29/07/2026,,,e_stage,,,,,,TRANSCRIPTOMIC,PolyA


#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [8]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,NA,,,6689,,,polyA,,,p_stage,SAMN02870963,,,,,,,"not clear if post larval should be considered final larval stage or first juvenile/sexually immature stage, dev stage states final larval stage. anat annotation should either be sexually immature organism or larva, I will annotate as larva for now however i am making a note in the lineage dev stages to review this, PMID: 25197823",,,,29/07/2026,,,p_stage,,,,,,TRANSCRIPTOMIC,PolyA
1,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,NA,,,6689,,,polyA,,,m_stage,SAMN02870962,,,,,,,"submitted new term request for mysis stage https://github.com/obophenotype/developmental-stage-ontologies/issues/124, PMID: 25197823",,,,29/07/2026,,,m_stage,,,,,,TRANSCRIPTOMIC,PolyA
2,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,,29/07/2026,,,z_stage,,,,,,TRANSCRIPTOMIC,PolyA
3,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,,29/07/2026,,,n_stage,,,,,,TRANSCRIPTOMIC,PolyA
4,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,,29/07/2026,,,e_stage,,,,,,TRANSCRIPTOMIC,PolyA


#### globin, replicates

In [9]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [10]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-29'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,NA,,,6689,,,polyA,,,p_stage,SAMN02870963,,,,,,,"not clear if post larval should be considered final larval stage or first juvenile/sexually immature stage, dev stage states final larval stage. anat annotation should either be sexually immature organism or larva, I will annotate as larva for now however i am making a note in the lineage dev stages to review this, PMID: 25197823",,,SAC,2026-07-29,,,p_stage,,,,,,TRANSCRIPTOMIC,PolyA
1,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,NA,,,6689,,,polyA,,,m_stage,SAMN02870962,,,,,,,"submitted new term request for mysis stage https://github.com/obophenotype/developmental-stage-ontologies/issues/124, PMID: 25197823",,,SAC,2026-07-29,,,m_stage,,,,,,TRANSCRIPTOMIC,PolyA
2,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,SAC,2026-07-29,,,z_stage,,,,,,TRANSCRIPTOMIC,PolyA
3,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,SAC,2026-07-29,,,n_stage,,,,,,TRANSCRIPTOMIC,PolyA
4,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,SAC,2026-07-29,,,e_stage,,,,,,TRANSCRIPTOMIC,PolyA


#### comments

In [ ]:
#library.loc[:,'comment'] = 'PMID: 25197823'

#### save complete file with correct columns

In [11]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,NA,,,6689,,,polyA,,,p_stage,SAMN02870963,,,,,,,"not clear if post larval should be considered final larval stage or first juvenile/sexually immature stage, dev stage states final larval stage. anat annotation should either be sexually immature organism or larva, I will annotate as larva for now however i am making a note in the lineage dev stages to review this, PMID: 25197823",,,SAC,2026-07-29
1,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,NA,,,6689,,,polyA,,,m_stage,SAMN02870962,,,,,,,"submitted new term request for mysis stage https://github.com/obophenotype/developmental-stage-ontologies/issues/124, PMID: 25197823",,,SAC,2026-07-29
2,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,SAC,2026-07-29
3,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,SAC,2026-07-29
4,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,SAC,2026-07-29


### experiment annotations

In [12]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP043546,Transcriptome of early development stages in Litopenaeus vannamei,Comparative Transcriptomic Characterization of the Early Development in Litopenaeus vannamei,SRA,,,,,,,PRJNA253518,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [13]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

5

In [14]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP043546,Transcriptome of early development stages in Litopenaeus vannamei,Comparative Transcriptomic Characterization of the Early Development in Litopenaeus vannamei,SRA,total,Bgee 1K,5,,,,PRJNA253518,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [15]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '25197823'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC4157780/'
experiment.loc[:,'DOI'] = '10.1371/journal.pone.0106201'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP043546,Transcriptome of early development stages in Litopenaeus vannamei,Comparative Transcriptomic Characterization of the Early Development in Litopenaeus vannamei,SRA,total,Bgee 1K,5,,,,PRJNA253518,25197823,https://pmc.ncbi.nlm.nih.gov/articles/PMC4157780/,10.1371/journal.pone.0106201,,


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [16]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [17]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [18]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 0
Errors: 0
Warnings: 0
Top codes:


#### check columns match

In [19]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [20]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
68640,SRX382954,SRP022057,Illumina HiSeq 2000,SRS418276,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,nauplii,nauplii,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,lv_n,SAMN02116197,,,,,,,10.1016/j.cbd.2014.07.001,,,SAC,2026-07-29
68641,SRX382953,SRP022057,Illumina HiSeq 2000,SRS418271,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,embryo,embryo,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,lv_e,SAMN02116196,,,,,,,10.1016/j.cbd.2014.07.001,,,SAC,2026-07-29
68642,SRX625475,SRP043546,Illumina HiSeq 2000,SRS646116,UBERON:0002548,larva,UBERON:0014858,crustacean post-larval stage,,whole body,postlarvae stage,other,not documented,perfect match,NA,,,6689,,,polyA,,,p_stage,SAMN02870963,,,,,,,not clear if post larval should be considered ...,,,SAC,2026-07-29
68643,SRX625474,SRP043546,Illumina HiSeq 2000,SRS646115,UBERON:0002548,larva,UBERON:0018378,crustacean larval stage,,whole body,mysis stage,perfect match,not documented,missing child term,NA,,,6689,,,polyA,,,m_stage,SAMN02870962,,,,,,,submitted new term request for mysis stage htt...,,,SAC,2026-07-29
68644,SRX625467,SRP043546,Illumina HiSeq 2000,SRS646108,UBERON:0002548,larva,UBERON:0014857,zoea stage,,whole body,zoea stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,z_stage,SAMN02870954,,,,,,,PMID: 25197823,,,SAC,2026-07-29
68645,SRX625466,SRP043546,Illumina HiSeq 2000,SRS646107,UBERON:0002548,larva,UBERON:0014406,nauplius stage,,whole body,nauplius stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,n_stage,SAMN02870953,,,,,,,PMID: 25197823,,,SAC,2026-07-29
68646,SRX625463,SRP043546,Illumina HiSeq 2000,SRS646105,UBERON:0000922,embryo,UBERON:0000068,embryo stage,,whole body,embryo stage,perfect match,not documented,perfect match,NA,,,6689,,,polyA,,,e_stage,SAMN02870946,,,,,,,PMID: 25197823,,,SAC,2026-07-29


In [21]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1315,SRP007829,smRNA sequencing of queen and virgin queen of ...,Deep sequencing of smRNA from queen and virgin...,SRA,total,Bgee 1K,4,,full_length,GSE31344,PRJNA154155,22885060,https://pmc.ncbi.nlm.nih.gov/articles/PMC3498763/,10.1016/j.cub.2012.07.042,,
1316,SRP022057,Litopenaeus vannamei Transcriptome or Gene exp...,Transcriptomic characterization of the early d...,SRA,partial,Bgee 1K,5,,,,PRJNA200996,,https://www.sciencedirect.com/science/article/...,10.1016/j.cbd.2014.07.001,,"no PMID, also rejected RT-PCR libraries"
1317,SRP043546,Transcriptome of early development stages in L...,Comparative Transcriptomic Characterization of...,SRA,total,Bgee 1K,5,,,,PRJNA253518,25197823,https://pmc.ncbi.nlm.nih.gov/articles/PMC4157780/,10.1371/journal.pone.0106201,,


### add annotations to git

In [22]:
! git pull

Already up to date.


In [23]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [24]:
! git add $git_experiment_path $git_library_path

In [25]:
! git commit -m $commit_message_exp

[develop d50210c] adding annotated bulk experiment SRP043546
 2 files changed, 6 insertions(+)


In [26]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.12 KiB | 2.12 MiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   9ecd6ff..d50210c  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push